In [ ]:
import re
import json
import numpy as np
import torch
import torch.nn.functional as F
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import (
    DistilBertTokenizer, DistilBertForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification
)
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords', quiet=True)
# ── CONFIG (must match your notebook exactly) 
EMOTION_LABELS = ['anger', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
MAX_LEN   = 100
MAX_WORDS = 10000
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ── CNN text preprocessing (same pipeline as training) 
KEEP_WORDS = {
    "not","no","never","none","nobody","nothing","neither","nor","cannot","without","lack",
    "very","too","so","extremely","highly","really","absolutely","completely","totally",
    "utterly","deeply","strongly","incredibly","super","quite","rather","especially",
    "slightly","somewhat","barely","hardly","scarcely","less","least",
    "but","however","though","although","yet","still",
    "feel","feels","felt","feeling","seem","seems",
    "maybe","perhaps","probably","definitely","certainly",
    "just","even","only","almost"
}
raw_stopwords     = set(stopwords.words('english'))
filtered_stopwords = raw_stopwords - KEEP_WORDS
stemmer = PorterStemmer()
def preprocess_for_cnn(text):
    # Stage 1: clean
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Stage 2: stopwords
    text = ' '.join([t for t in text.split() if t not in filtered_stopwords])
    # Stage 3: stem
    text = ' '.join([stemmer.stem(w) for w in text.split()])
    return text
# ── Load all models (run once) 
def load_models():
    # DistilBERT
    db_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    db_model     = DistilBertForSequenceClassification.from_pretrained(
                       "distilbert-base-uncased", num_labels=len(EMOTION_LABELS))
    db_model.load_state_dict(torch.load("best_distilbert.pt", map_location=DEVICE))
    db_model.to(DEVICE).eval()
    # RoBERTa
    rb_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    rb_model     = RobertaForSequenceClassification.from_pretrained(
                       "roberta-base", num_labels=len(EMOTION_LABELS))
    rb_model.load_state_dict(torch.load("best_roberta.pt", map_location=DEVICE))
    rb_model.to(DEVICE).eval()
    # CNN
    with open("cnn_tokenizer.json", "r") as f:
        cnn_tokenizer = keras.preprocessing.text.tokenizer_from_json(f.read())
    cnn_model = keras.models.load_model("best_cnn.h5")
    return db_tokenizer, db_model, rb_tokenizer, rb_model, cnn_tokenizer, cnn_model
# ── Compare sentences across all models 
def compare(sentences: list):
    db_tok, db_mod, rb_tok, rb_mod, cnn_tok, cnn_mod = load_models()
    for sentence in sentences:
        print(f"\n{'='*65}")
        print(f'Input: "{sentence}"')
        print(f"{'='*65}")
        results = {}
        # DistilBERT
        enc = db_tok(sentence, return_tensors="pt", truncation=True,
                     max_length=128, padding=True).to(DEVICE)
        with torch.no_grad():
            logits = db_mod(**enc).logits
        results["DistilBERT"] = dict(zip(EMOTION_LABELS,
                                F.softmax(logits, dim=-1).squeeze().cpu().numpy()))
        # RoBERTa
        enc = rb_tok(sentence, return_tensors="pt", truncation=True,
                     max_length=128, padding=True).to(DEVICE)
        with torch.no_grad():
            logits = rb_mod(**enc).logits
        results["RoBERTa"] = dict(zip(EMOTION_LABELS,
                             F.softmax(logits, dim=-1).squeeze().cpu().numpy()))
        # CNN — apply same preprocessing as training
        processed = preprocess_for_cnn(sentence)
        seq = cnn_tok.texts_to_sequences([processed])
        seq = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
        results["CNN"] = dict(zip(EMOTION_LABELS,
                          cnn_mod.predict(seq, verbose=0).squeeze()))
        # Print table
        print(f"{'Emotion':<12}", end="")
        for name in results:
            print(f"{name:>15}", end="")
        print()
        print("-" * 57)
        for label in EMOTION_LABELS:
            print(f"{label:<12}", end="")
            for probs in results.values():
                print(f"{probs[label]:>14.1%}", end="")
            print()
        print("-" * 57)
        print(f"{'Predicted':<12}", end="")
        for name, probs in results.items():
            winner = max(probs, key=probs.get)
            print(f"{winner:>15}", end="")
        print()

In [ ]:
sentences = [
    "I can't believe how happy I am today!",
    "This makes me so angry I could scream.",
    "I'm scared of what might happen next."
]
compare(sentences)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
++-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
++-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: "I can't believe how happy I am today!"
Emotion          DistilBERT        RoBERTa            CNN
anger                 0.0%          0.0%          0.0%
fear                  0.0%          0.0%          0.0%
joy                 100.0%        100.0%         97.8%
neutral               0.0%          0.0%          0.0%
sadness               0.0%          0.0%          0.4%
surprise              0.0%          0.0%          1.7%
Predicted               joy            joy            joy
Input: "This makes me so angry I could scream."
Emotion          DistilBERT        RoBERTa            CNN
anger               100.0%        100.0%          0.0%
fear                  0.0%          0.0%          0.0%
joy                   0.0%          0.0%        100.0%
neutral               0.0%          0.0%          0.0%
sadness               0.0%          0.0%          0.0%
surprise              0.0%          0.0%          0.0%
Predicted             anger          anger            joy
Input: "I'm s

In [ ]:
sentences = [
    # --- ANGER (transformers often confuse with sadness) ---
    "I can't believe they lied to me again, this is absolutely unacceptable",
    "This is so unfair, I'm done with all of this nonsense",
    # --- FEAR (often confused with sadness or surprise) ---
    "I have a really bad feeling something terrible is about to happen",
    "I can't stop shaking, I'm terrified of what comes next",
    # --- JOY (sometimes confused with surprise) ---
    "I just got the job offer, I'm so incredibly happy right now",
    "Today was absolutely perfect, couldn't have asked for more",
    # --- NEUTRAL (hardest class — all models struggle here) ---
    "I went to the store and bought some groceries",
    "The meeting is scheduled for 3pm tomorrow",
    # --- SADNESS (often bleeds into fear or anger) ---
    "I miss them so much, everything just feels empty without them",
    "Nobody reached out, I spent the whole day completely alone",
    # --- SURPRISE (rarest class, most likely to be misclassified) ---
    "I had no idea they were planning this, I'm completely speechless",
    "Wait, that actually happened? I genuinely cannot believe it",
]
compare(sentences)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
++-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
++-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: "I can't believe they lied to me again, this is absolutely unacceptable"
Emotion          DistilBERT        RoBERTa            CNN
anger               100.0%        100.0%          0.4%
fear                  0.0%          0.0%          0.5%
joy                   0.0%          0.0%         76.2%
neutral               0.0%          0.0%          0.4%
sadness               0.0%          0.0%          7.9%
surprise              0.0%          0.0%         14.6%
Predicted             anger          anger            joy
Input: "This is so unfair, I'm done with all of this nonsense"
Emotion          DistilBERT        RoBERTa            CNN
anger               100.0%        100.0%          3.1%
fear                  0.0%          0.0%          0.2%
joy                   0.0%          0.0%          5.4%
neutral               0.0%          0.0%          0.4%
sadness               0.0%          0.0%         90.8%
surprise              0.0%          0.0%          0.2%
Predicted             a

In [ ]:
neutral_sentences = [
    # Pure factual statements
    "The train arrives at 6:45 in the morning",
    "She sent the document to the team yesterday",
    "The package weighs approximately three kilograms",
    "He closed the laptop and left the room",
    "The report has twelve pages",
    # Task / calendar language
    "reminder to call the dentist on monday",
    "need to buy milk eggs and bread",
    "the file is saved in the downloads folder",
    # Totally emotionless observations
    "it rained this afternoon",
    "the tv was on in the background",
]
compare(neutral_sentences)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
++-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
++-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: "The train arrives at 6:45 in the morning"
Emotion          DistilBERT        RoBERTa            CNN
anger                 0.0%          0.0%          1.5%
fear                  0.0%          0.0%          0.9%
joy                 100.0%        100.0%         73.2%
neutral               0.0%          0.0%          0.2%
sadness               0.0%          0.0%         23.6%
surprise              0.0%          0.0%          0.6%
Predicted               joy            joy            joy
Input: "She sent the document to the team yesterday"
Emotion          DistilBERT        RoBERTa            CNN
anger                 0.1%          0.0%          3.1%
fear                  0.5%          0.0%         28.0%
joy                   1.2%        100.0%         38.5%
neutral              98.0%          0.0%          1.2%
sadness               0.1%          0.0%          6.1%
surprise              0.1%          0.0%         23.2%
Predicted           neutral            joy            joy
Input